# شام — الدفتر الشامل (مقعد واحد بدل ثلاثة دفاتر)

يجمع الدفاتر التي لا تُجدوَل في مكان آخر: **المدرّب الحي** (زحف وجمع وترميز وتدريب)، و**المرحلة الأولى** (تدريب النص)، و**المرحلة الثانية** (الصورة والصوت مع النص). كل جلسة تشغّل ما يصلح لوقتها: يدور الترتيب يومياً فينال كل دفتر دوره، ويشغَّل دفتر ثانٍ في الجلسة ذاتها فقط إن كانت ميزانيته الزمنية تتسع.

لا يكرّر ما تعمله دفاتر أخرى: دفتر الصورة يشغّل الصوت والفيديو بعده، والمسار A والمسار B والمحادثة تبقى كلٌّ في مقعدها (كاتب واحد لكل مجموعة بيانات).

- **المسرّع:** CPU فيُجدوَل يومياً، أو GPU فيُشغَّل يدوياً (كاكل يرفض جدولة دفتر عليه GPU).
- **Internet:** يجب أن يكون مفعّلاً.
- **الأسرار:** KAGGLE_USERNAME و KAGGLE_KEY و GITHUB_TOKEN (نفس بقية الدفاتر). لا يحتاج أي Add Input.
- **الاستئناف:** كل دفتر داخله يستأنف تلقائياً من آخر نقطة منشورة، فلا يضيع شيء بين جلسة وأخرى.
- كل تطوير لاحق يصل عبر المستودع دون إعادة استيراد.


## 1) سحب الكود الحقيقي من GitHub

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("الكود موجود بالفعل في هذه الجلسة — تم سحب أي تحديثات جديدة عليه.")

# أمان: git يحفظ رابط الاستنساخ (بما فيه GITHUB_TOKEN) حرفياً داخل
# .git/config -- وهذا المجلد يبقى ضمن نتاج (Output) هذه الجلسة، الذي قد
# يُستخدَم لاحقاً كمُدخَل (Notebook Output) لجلسة أخرى، أو يُشارَك بأي شكل.
# نزع التوكن من الرابط المحفوظ فور نجاح الاستنساخ يمنع تسربه عبر هذا
# المسار تماماً (ثغرة حقيقية اكتشفتها المالكة، 2026-09-21).
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", "https://github.com/jonsnow-org/Ttbik.git"], check=True)

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "sham_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود ShamSmall الحقيقي جاهز في:", CODE_DIR)


## 2) تثبيت المكتبات

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "av", "soundfile", "tokenizers", "datasets", "kaggle"], check=False)
print("تم تجهيز المكتبات.")


## 3) ما يعمل اليوم
الدفاتر المختارة تعمل بالترتيب، كلٌّ بخلاياه ومواعيده الداخلية وميزانيته. فشل دفتر لا يوقف الذي بعده، وتُطبع نتيجة كل واحد في النهاية.

In [ ]:
import datetime
import torch
import sham_inputs  # يثبّت إصلاحات التدريب المشتركة ويطبّق تحديثات الخلايا
from sham_chain import ALL_IN_ONE_STAGES, pick_stages, run_notebooks

CUDA = torch.cuda.is_available()
STAGES_TODAY = pick_stages(datetime.date.today().toordinal(), 11.0, CUDA)
# يدوياً لتشغيل دفتر بعينه: STAGES_TODAY = ["sham_small_training"]   (الخيارات: ALL_IN_ONE_STAGES)
print("الجهاز:", "GPU" if CUDA else "CPU", "| اليوم يعمل:", STAGES_TODAY)

CHAIN_RESULTS = run_notebooks(STAGES_TODAY, code_dir=CODE_DIR)
print("نتيجة الجلسة:", CHAIN_RESULTS)
